In [3]:
from game import Game
import time
import os
import re

## Load maps

In [4]:
import re
import os
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = [1, 2, 3]

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map_index, map_file_indices, method, SOLVERS):
    if map_index not in map_file_indices:
        print(f"Invalid map index: {map_index}.")
        return False
    if method not in SOLVERS:
        print(f"Invalid method: {method}.")
        return False
    return True
def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [5]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [6]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [7]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [8]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [9]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [10]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [11]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

### BFS

In [12]:
# TODO: Must return moves (if there is no solution return None), number of visited states
from collections import deque
def solver_bfs(game_map):
    game = Game(game_map)
    if game.is_game_won():
        return ("", 1)


    start_state = (
        game.get_player_position(),
        tuple(sorted(game.get_box_locations()))
    )


    queue = deque([(start_state, "")])
    visited = set([start_state])
    visited_count = 0

    while queue:
        (player_pos, boxes), path = queue.popleft()
        visited_count += 1

        cloned_game = game.clone()
        cloned_game.set_player_position(player_pos)
        cloned_game.set_box_positions(boxes)

        if cloned_game.is_game_won():
            return (path, visited_count)

        for move in ['U', 'D', 'L', 'R']:
            next_game = cloned_game.clone()
            if next_game.apply_move(move):
                new_state = (
                    next_game.get_player_position(),
                    tuple(sorted(next_game.get_box_locations()))
                )

                if new_state not in visited:
                    visited.add(new_state)
                    queue.append((new_state, path + move))

    return (None, visited_count)
    return None, 0

### DFS

In [13]:
# TODO: Must return moves, number of visited states
from collections import deque
def solver_dfs(game_map):
    game = Game(game_map)
    if game.is_game_won():
        return ("", 1)

    start_state = (
        game.get_player_position(),
        tuple(sorted(game.get_box_locations()))
    )

    #for storing
    queue = deque([(start_state, "")])

    visited = set([start_state])

    visited_count = 0


    while queue:
        (player_pos, boxes), path = queue.popleft()
        visited_count += 1


        cloned_game = game.clone()
        cloned_game.set_player_position(player_pos)
        cloned_game.set_box_positions(boxes)

        if cloned_game.is_game_won():
            return (path, visited_count)

        for move in ['U', 'D', 'L', 'R']:
            next_game = cloned_game.clone()
            if next_game.apply_move(move):
                new_state = (
                    next_game.get_player_position(),
                    tuple(sorted(next_game.get_box_locations()))
                )

                if new_state not in visited:
                    visited.add(new_state)
                    queue.append((new_state, path + move))

    return (None, visited_count, visited)
    return None, 0

### IDS

In [14]:
# TODO: Must return moves, number of visited states
def solver_ids(game_map , max_depth=100):
    game = Game(game_map)
    if game.is_game_won():
        return ("", 1)


    visited_count = 0

    def dls(current_game, path, depth, limit, visited):

        nonlocal visited_count
        visited_count += 1

        if current_game.is_game_won():
            return path

        if depth == limit:
            return None

        state = (
            current_game.get_player_position(),
            tuple(sorted(current_game.get_box_locations()))
        )
        visited.add(state)

        for move in ['U', 'D', 'L', 'R']:
            next_game = current_game.clone()
            if next_game.apply_move(move):
                new_state = (
                    next_game.get_player_position(),
                    tuple(sorted(next_game.get_box_locations()))
                )
                if new_state not in visited:
                    result = dls(next_game, path + move, depth + 1, limit, visited)
                    if result is not None:
                        return result

        return None

    for limit in range(max_depth + 1):
        visited_this_iteration = set()

        start_clone = game.clone()

        result = dls(start_clone, "", 0, limit, visited_this_iteration)
        if result is not None:
            return (result, visited_count)

    return (None, visited_count)
    return None, 0

### A*

### f = g + weight * h
### g = number of moves agent gotta use

In [15]:
# TODO
import heapq

def heuristic(game_map):
    return 0
# TODO: Must return moves, number of visited states
def solver_astar(game_map, heuristic_func=heuristic, weight=1):

    game = Game(game_map)

    if game.is_game_won():
        return ("", 1)

    start_state = (
        game.get_player_position(),
        tuple(sorted(game.get_box_locations()))
    )

    def compute_heuristic(player_pos, boxes):
        cloned = game.clone()
        cloned.set_player_position(player_pos)
        cloned.set_box_positions(boxes)
        return heuristic_func(cloned)

    open_list = []
    visited = {}
    visited_count = 0

    h0 = compute_heuristic(*start_state)
    f0 = 0 + weight * h0
    heapq.heappush(open_list, (f0, 0, start_state, ""))
    visited[start_state] = 0

    while open_list:
        f, g, (player_pos, boxes), path = heapq.heappop(open_list)
        visited_count += 1

        if visited[(player_pos, boxes)] < g:
            continue

        current_clone = game.clone()
        current_clone.set_player_position(player_pos)
        current_clone.set_box_positions(boxes)

        if current_clone.is_game_won():
            return (path, visited_count)

        for move in ['U', 'D', 'L', 'R']:
            neighbor_clone = current_clone.clone()
            if neighbor_clone.apply_move(move):
                new_state = (
                    neighbor_clone.get_player_position(),
                    tuple(sorted(neighbor_clone.get_box_locations()))
                )
                new_g = g + 1

                if new_state not in visited or visited[new_state] > new_g:
                    visited[new_state] = new_g
                    h_val = compute_heuristic(*new_state)
                    new_f = new_g + weight * h_val
                    heapq.heappush(open_list, (new_f, new_g, new_state, path + move))

    return (None, visited_count)


## Solve

In [16]:
SOLVERS = {
    "BFS": Game.bfs_solve,
    "DFS": Game.dfs_solve,
    "IDS": Game.ids_solve,
    "A*": Game.astar_solve
}

In [17]:
def solve(map_index, method):
    if not is_valid_input(map_index, map_file_indices, method, SOLVERS):
        return

    file_name = f"map{map_index}.txt"
    with open(f"./assets/maps/{file_name}") as f:
        game_map = f.read()

    game_instance = Game(game_map)

    import time
    start_time = time.time()
    moves = SOLVERS[method](game_instance)
    end_time = time.time()

    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map_index}.")

    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [24]:
solve(4, "BFS") # Solve map 1 using BFS

BFS took 0.0 seconds on map 4.
No Solution Found!


In [19]:
def solve_all():
    for map_index in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map_index, method)

In [20]:
solve_all() # Solve all maps using all methods

BFS took 0.01 seconds on map 1.
No Solution Found!
DFS took 0.01 seconds on map 1.
No Solution Found!
IDS took 0.3 seconds on map 1.
No Solution Found!
A* took 0.01 seconds on map 1.
No Solution Found!
BFS took 0.0 seconds on map 2.
6 moves were used: LUDDUL
DFS took 0.0 seconds on map 2.
6 moves were used: LUDDUL
IDS took 0.01 seconds on map 2.
6 moves were used: LUDDUL
A* took 0.0 seconds on map 2.
6 moves were used: LDULRU
BFS took 0.01 seconds on map 3.
No Solution Found!
DFS took 0.02 seconds on map 3.
No Solution Found!
IDS took 0.66 seconds on map 3.
No Solution Found!
A* took 0.02 seconds on map 3.
No Solution Found!
BFS took 0.0 seconds on map 4.
No Solution Found!
DFS took 0.0 seconds on map 4.
No Solution Found!
IDS took 0.01 seconds on map 4.
No Solution Found!
A* took 0.0 seconds on map 4.
No Solution Found!
BFS took 0.26 seconds on map 5.
15 moves were used: ULDDRDLLLUUURUL
DFS took 0.04 seconds on map 5.
137 moves were used: UULDDUULDDDDLUUUDDDLUUUURDDDDRUUURULDDDDRUUDDL

KeyboardInterrupt: 